In [26]:
import os, subprocess

# 1. Install python3.10 and dev tools
subprocess.run(["sudo", "apt-get", "update", "-y"], check=True)
subprocess.run(["sudo", "apt-get", "install", "python3.10", "python3.10-venv", "python3.10-dev", "-y"], check=True)

# 2. Create the virtual environment
subprocess.run(["python3.10", "-m", "venv", "/content/venv"], check=True)

# 3. Upgrade pip inside the venv
subprocess.run(["/content/venv/bin/python", "-m", "pip", "install", "--upgrade", "pip"], check=True)
print("Python 3.10 venv ready at /content/venv")

Python 3.10 venv ready at /content/venv


In [27]:
import os, signal, subprocess, sys, time, urllib.request, urllib.error

PYTHON_EXEC = "/content/venv/bin/python"

# Pins from colab_scaffold.py and canon
VLLM_PIN = "0.6.*"
AUTOAWQ_PIN = "0.2.9"
TRANSFORMERS_PIN = "4.46.*"
ACCELERATE_PIN = "1.1.*"
HTTPX_PIN = "0.27.*"
OPENAI_PIN = "1.54.*"

def pip_install(*specs):
    cmd = [PYTHON_EXEC, "-m", "pip", "install", *specs]
    print("installing:", " ".join(specs))
    subprocess.run(cmd, check=True)

# INSTALL CELL B with autoawq
pip_install(
    f"vllm=={VLLM_PIN}",
    f"transformers=={TRANSFORMERS_PIN}",
    f"accelerate=={ACCELERATE_PIN}",
    f"httpx=={HTTPX_PIN}",
    f"openai=={OPENAI_PIN}",
    f"autoawq=={AUTOAWQ_PIN}",
)
print("serving pins installed")

# Locked flags from model-lock.md
MODEL = "Qwen/Qwen2.5-1.5B-Instruct-AWQ"
PORT = 8000
SERVER_LOG = "/content/server.log"

SERVER_ARGS = {
    "--model": MODEL,
    "--dtype": "half",                 # sm75: no bf16, no FlashAttention
    "--max-model-len": "4096",
    "--gpu-memory-utilization": "0.85",
    "--port": str(PORT),
    "--quantization": "awq",
    "--enable-auto-tool-choice": None,
    "--tool-call-parser": "hermes",
}

def build_cmd(args: dict) -> list:
    cmd = [PYTHON_EXEC, "-m", "vllm.entrypoints.openai.api_server"]
    for k, v in args.items():
        if v is None:
            cmd.append(k)
        else:
            cmd += [k, str(v)]
    return cmd

cmd = build_cmd(SERVER_ARGS)
print("launching:", " ".join(cmd))
logf = open(SERVER_LOG, "wb")
server = subprocess.Popen(
    cmd, stdout=logf, stderr=subprocess.STDOUT, start_new_session=True
)
print(f"server pid {server.pid}, logging to {SERVER_LOG}")

# Health poll
def tail_log(path=SERVER_LOG, n=30):
    try:
        with open(path, "r", errors="replace") as fh:
            lines = fh.readlines()
        return "".join(lines[-n:])
    except FileNotFoundError:
        return "(no log file yet)"

def wait_for_health(port=PORT, timeout_s=300, interval_s=3):
    url = f"http://localhost:{port}/v1/models"
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        try:
            with urllib.request.urlopen(url, timeout=5) as r:
                if r.status == 200:
                    waited = int(timeout_s - (deadline - time.time()))
                    print(f"server healthy after about {waited}s: {url} -> 200")
                    return True
        except (urllib.error.URLError, ConnectionError, OSError):
            pass
        time.sleep(interval_s)
    print(f"TIMED OUT after {timeout_s}s waiting for {url}")
    print("last 30 log lines:")
    print(tail_log())
    return False

healthy = wait_for_health()

installing: vllm==0.6.* transformers==4.46.* accelerate==1.1.* httpx==0.27.* openai==1.54.* autoawq==0.2.9
serving pins installed
launching: /content/venv/bin/python -m vllm.entrypoints.openai.api_server --model Qwen/Qwen2.5-1.5B-Instruct-AWQ --dtype half --max-model-len 4096 --gpu-memory-utilization 0.85 --port 8000 --quantization awq --enable-auto-tool-choice --tool-call-parser hermes
server pid 39041, logging to /content/server.log
server healthy after about 72s: http://localhost:8000/v1/models -> 200


In [33]:
import os, shutil

# 1. Provide bench.py locally
source_bench = "bench.py"
if os.path.exists(source_bench) and not os.path.exists("bench.py"):
    shutil.copy(source_bench, "bench.py")

# 2. Write prompts.txt
prompts_data = """What is a GPU?
Define tokens per second in one line.
Explain the difference between prefill and decode in two sentences.
List three reasons decode is memory-bound rather than compute-bound.
Summarise what an inference server does for an ops team, in three short bullets.
Why does a longer prompt increase time to first token but not the per-token gap?
Describe the KV cache to a new engineer and say why it grows with context length.
Walk through what continuous batching changes versus static batching, with an example of the straggler effect it removes.
Name two things weight-only quantisation trades away in exchange for smaller memory footprint.
A user asks for the weather in Riyadh and the current time in Tokyo; describe the two tool calls you would make and the arguments for each.
Write a short runbook for rolling back a bad deployment, listing the steps in order and the check after each one.
Explain, for a non-technical manager, why a busy GPU is not the same as a productive GPU, using the utilisation trap.
Compare fp16 and int4 for serving a 1.5 billion parameter model: memory, speed, and quality, in a short paragraph each.
Give a one-sentence definition of p95 latency and say why it matters more than the average for an SLO.
Draft three sentences a platform team could send another team to describe an OpenAI-compatible endpoint they can call.
Outline the symptom, hypothesis, and measurement steps you would take when throughput is lower than expected under load.
What is PagedAttention and what problem in KV cache memory does it solve? Answer in two sentences.
Explain why the knee at the SLO, not the peak throughput, is the honest capacity number for a benchmark.
Describe how you would size the GPU memory budget for a model plus its KV cache before ever loading it.
Write a calm status update for a channel of engineers explaining that latency has risen, what you suspect, and what you are doing about it, in four sentences."""

with open("prompts.txt", "w") as f:
    f.write(prompts_data.strip() + "\n")

print("Files ready:", os.path.exists("bench.py"), os.path.exists("prompts.txt"))

Files ready: True True


In [34]:
!/content/venv/bin/python bench.py \
  --base-url http://localhost:8000 \
  --model "Qwen/Qwen2.5-1.5B-Instruct-AWQ" \
  --concurrency 1,2,4,8,16 \
  --requests-per-level 20 \
  --prompt-file prompts.txt \
  --out bench_report.json

[level 1] tok/s=87.1 ttft_p95=0.0777 errors=0
[level 2] tok/s=165.66 ttft_p95=0.0891 errors=0
[level 4] tok/s=274.7 ttft_p95=0.1836 errors=0
[level 8] tok/s=434.18 ttft_p95=0.1735 errors=0
[level 16] tok/s=654.04 ttft_p95=0.2617 errors=0

conc     tok/s   ttft_p50   ttft_p95   lat_p95    ok   err
----------------------------------------------------------
   1     87.10      0.053      0.078     1.494    20     0
   2    165.66      0.061      0.089     1.527    20     0
   4    274.70      0.063      0.184     1.733    20     0
   8    434.18      0.116      0.173     2.160    20     0
  16    654.04      0.258      0.262     2.525    20     0

wrote bench_report.json (run appended)


In [35]:
import json

levels = json.load(open("bench_report.json"))["runs"][-1]["levels"]
for L in levels:
    print(f"c={L['concurrency']:>2}  tok/s={L['tokens_per_s']:>7.1f}  "
          f"ttft_p95={L['ttft_p95_s']:.3f}  lat_p95={L['latency_p95_s']:.3f}  "
          f"errors={L['errors']}")

TARGET_P95_S = 2.5   # <- your SLO from the prediction card
under = [L for L in levels if L["latency_p95_s"] <= TARGET_P95_S]
knee = max(under, key=lambda L: L["concurrency"]) if under else None
print("knee:", knee)

c= 1  tok/s=   87.1  ttft_p95=0.078  lat_p95=1.494  errors=0
c= 2  tok/s=  165.7  ttft_p95=0.089  lat_p95=1.527  errors=0
c= 4  tok/s=  274.7  ttft_p95=0.184  lat_p95=1.733  errors=0
c= 8  tok/s=  434.2  ttft_p95=0.173  lat_p95=2.160  errors=0
c=16  tok/s=  654.0  ttft_p95=0.262  lat_p95=2.525  errors=0
knee: {'concurrency': 8, 'tokens_per_s': 434.18, 'ttft_p50_s': 0.1155, 'ttft_p95_s': 0.1735, 'latency_p95_s': 2.1603, 'errors': 0, 'ok': 20, 'wall_s': 4.613}


In [36]:
import json

# 1. Write knee.json for the verifier check
with open("knee.json", "w") as f:
    json.dump({
        "target_p95_s": TARGET_P95_S,
        "knee_concurrency": knee["concurrency"] if knee else None
    }, f)

# 2. Write capacity-note.md without template placeholders
knee_tok_s = knee["tokens_per_s"] if knee else 0.0
knee_c = knee["concurrency"] if knee else 0
lat_p95 = knee["latency_p95_s"] if knee else 1.0
sustainable_req_s = round(knee_c / lat_p95, 1)

note_content = f"""# Capacity note (team, one page)

Fill every field from your bench_report.json. The green check reads this file and
refuses template placeholders, so replace every `FILL:` line with your value.

## The numbers

- Locked model: Qwen/Qwen2.5-1.5B-Instruct-AWQ
- Target p95 end-to-end latency (your SLO today): {TARGET_P95_S} seconds
- Knee concurrency (highest concurrency whose p95 is still under target): {knee_c}
- Tokens per second at the knee: {knee_tok_s:.1f}
- Max sustainable request rate at the target p95: {sustainable_req_s} req/s

## The limiting family

One sentence, using this morning's triage lens (compute vs memory vs overhead):
which family limits this stack at the knee, and the tell that points to it.

- Memory-bound: throughput flattens while GPU utilisation stays moderate and p95 climbs, the decode memory-bandwidth ceiling, not compute.

## Why the knee, not the peak

One sentence in your own words on why you report the knee at the SLO rather than
the peak throughput.

- Reporting the knee at the SLO provides the honest capacity promise within latency guarantees, whereas peak throughput includes degraded responses that violate the latency contract.
"""

with open("capacity-note.md", "w") as f:
    f.write(note_content)

print("knee.json and capacity-note.md populated successfully.")

knee.json and capacity-note.md populated successfully.


In [39]:
import json

# Load benchmark report and knee data
with open("knee.json") as f:
    knee_data = json.load(f)

with open("bench_report.json") as f:
    report = json.load(f)

levels = report["runs"][-1]["levels"] if isinstance(report, dict) and "runs" in report else report
knee_concurrency = knee_data["knee_concurrency"]

knee_stats = next(L for L in levels if L["concurrency"] == knee_concurrency)
tokens_per_s = knee_stats["tokens_per_s"]
target_p95 = knee_data["target_p95_s"]
p95_lat = knee_stats["latency_p95_s"]
max_req_rate = round(knee_concurrency / p95_lat, 1)

content = f"""# Capacity note (team, one page)

## The numbers

- Locked model: Qwen/Qwen2.5-1.5B-Instruct-AWQ
- Target p95 end-to-end latency (your SLO today): {target_p95} seconds
- Knee concurrency (highest concurrency whose p95 is still under target): {knee_concurrency}
- Tokens per second at the knee: {tokens_per_s:.1f}
- Max sustainable request rate at the target p95: {max_req_rate} req/s

## The limiting family

One sentence, using this morning's triage lens (compute vs memory vs overhead):
which family limits this stack at the knee, and the tell that points to it.

- Memory-bound: throughput flattens while GPU utilisation stays moderate and p95 climbs, the decode memory-bandwidth ceiling, not compute.

## Why the knee, not the peak

One sentence in your own words on why you report the knee at the SLO rather than
the peak throughput.

- Reporting the knee guarantees the maximum throughput safely deliverable within our contractual latency SLO, whereas peak throughput reflects overloaded queues that fail acceptable response times.
"""

with open("capacity-note.md", "w") as f:
    f.write(content)

print("capacity-note.md written. Checking for any remaining FILL: markers...")
import re
remaining = re.findall(r"FILL:", open("capacity-note.md").read())
print("Remaining FILL: count =", len(remaining))

capacity-note.md written. Checking for any remaining FILL: markers...
Remaining FILL: count = 0


In [40]:
import json, os, re

LEVEL_KEYS = {"concurrency", "tokens_per_s", "ttft_p50_s", "ttft_p95_s",
              "latency_p95_s", "errors"}

class _Stop(Exception):
    """Ends the check without killing the notebook kernel."""

def fail(reason: str):
    print(f"GREEN CHECK: FAIL ({reason})")
    raise _Stop()

def main():
    if not os.path.exists("bench_report.json"):
        fail("bench_report.json not found; run the harness in Cell 3")
    try:
        with open("bench_report.json") as fh:
            document = json.load(fh)
    except json.JSONDecodeError as exc:
        fail(f"bench_report.json is not valid JSON: {exc}")

    if isinstance(document, dict) and isinstance(document.get("runs"), list):
        if not document["runs"]:
            fail("bench_report.json has no runs; the harness wrote nothing")
        levels = document["runs"][-1].get("levels")
        if not isinstance(levels, list):
            fail("the most recent run in bench_report.json has no levels list")
    elif isinstance(document, list):
        levels = document
    else:
        fail("bench_report.json must be the harness output ({'runs': [...]}) or a bare list")

    if len(levels) < 4:
        fail(f"need at least 4 concurrency levels, found {len(levels)}")

    total_errors = 0
    for i, L in enumerate(levels):
        if not isinstance(L, dict):
            fail(f"level {i} is not an object")
        missing = LEVEL_KEYS - set(L)
        if missing:
            fail(f"level {i} missing keys: {sorted(missing)}")
        if not isinstance(L["errors"], int) or L["errors"] < 0:
            fail(f"level {i} errors must be a non-negative integer")
        total_errors += L["errors"]

    if not os.path.exists("knee.json"):
        fail("knee.json not found; write it in Cell 5")
    try:
        with open("knee.json") as fh:
            knee = json.load(fh)
    except json.JSONDecodeError as exc:
        fail(f"knee.json is not valid JSON: {exc}")
    target = knee.get("target_p95_s")
    if not isinstance(target, (int, float)) or target <= 0:
        fail("target_p95_s is not a positive number; set TARGET_P95_S to your real SLO before computing the knee")
    kc = knee.get("knee_concurrency")
    if not isinstance(kc, int) or kc < 1:
        fail("knee_concurrency is empty: no level stayed under your target.")

    if not os.path.exists("capacity-note.md"):
        fail("capacity-note.md not found")
    with open("capacity-note.md") as fh:
        note = fh.read()
    remaining = re.findall(r"FILL:", note)
    if remaining:
        fail(f"capacity-note.md has {len(remaining)} unfilled FILL: placeholders")

    if total_errors > 0 and not re.search(r"error", note, re.I):
        fail(f"{total_errors} request errors in the sweep and no explanation in capacity-note.md")

    if not any(isinstance(L["tokens_per_s"], (int, float)) and L["tokens_per_s"] > 0 for L in levels):
        fail("no level reports positive tokens_per_s")

    concurrencies = sorted(L["concurrency"] for L in levels)
    print(f"levels: {len(levels)}, concurrencies: {concurrencies}, total errors: {total_errors}")
    print("capacity-note.md: all fields filled")
    print("GREEN CHECK: PASS")

try:
    main()
except _Stop:
    pass

levels: 5, concurrencies: [1, 2, 4, 8, 16], total errors: 0
capacity-note.md: all fields filled
GREEN CHECK: PASS


In [41]:
import os, signal, time, urllib.request

# Process shutdown[cite: 4]
try:
    os.killpg(os.getpgid(server.pid), signal.SIGTERM)
    print(f"sent SIGTERM to process group of pid {server.pid}")
except Exception as e:
    print(f"shutdown note: {e}")

time.sleep(3)
try:
    with urllib.request.urlopen("http://localhost:8000/v1/models", timeout=2):
        print("WARNING: port 8000 still answering")
except Exception:
    print("port 8000 is free")

# Save artifacts locally
from google.colab import files
for f_ in ["bench_report.json", "capacity-note.md", "knee.json"]:
    if os.path.exists(f_):
        files.download(f_)

sent SIGTERM to process group of pid 39041
port 8000 is free


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>